In [1]:
# %pip install -U langchain-text-splitters

### RAG PIPELINE - Data Ingestion to Vector DB Pipeline

In [2]:
import os
from pathlib import Path
# from lang.document_loaders import PyMuPDFLoader, PyPDFLoader
# from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_classic.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter

c:\Users\ShubhanshuJha\Downloads\RAG_Project\myvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def read_pdfs(pdf_dir_path: str):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_dir_path)
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"(*) Found {len(pdf_files)} pdf files to process.")

    for pdf_file in pdf_files:
        print(f"(*) Reading: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source info to the metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
                doc.metadata['processed_by'] = 'Shubhanshu'
            all_documents.extend(documents)
            print(f"(*) Loaded data from {len(documents)} pages.")
        except Exception as ex:
            print(f"(*) Error while reading {pdf_file.name} file -- {ex}")
    print(f"(*) Total documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the 'data' directory
all_pdf_documents = read_pdfs("../data")

(*) Found 4 pdf files to process.
(*) Reading: amazon_kinesis_interview_qna.pdf
(*) Loaded data from 1 pages.
(*) Reading: apache_kafka_interview_qna.pdf
(*) Loaded data from 2 pages.
(*) Reading: aws_glue_interview_qna.pdf
(*) Loaded data from 2 pages.
(*) Reading: aws_lambda_interview_qna.pdf
(*) Loaded data from 1 pages.
(*) Total documents loaded: 6


In [4]:
all_pdf_documents

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-08-27T11:17:36+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-08-27T11:17:36+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf_files\\amazon_kinesis_interview_qna.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'amazon_kinesis_interview_qna.pdf', 'file_type': 'pdf', 'processed_by': 'Shubhanshu'}, page_content="Amazon Kinesis Interview Questions and Answers\nInterview-style questions covering Kinesis Data Streams, Kinesis Data Firehose, shards, partition keys,\nconsumer patterns, and how Kinesis compares to other streaming platforms.\nQ1. What is Amazon Kinesis?\nAmazon Kinesis is a family of managed services for collecting, processing, and analyzing real-time streaming\ndata at scale. It includes Kinesis Data Streams, Kinesis Data Firehose, and Kinesis Data Analytics, eac

In [5]:
### Creating Data Chunks
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=['\n\n', '\n', ' ', '']
    )

    split_doc_chunks = text_splitter.split_documents(documents)
    print(f"(*) Split {len(documents)} into {len(split_doc_chunks)} chunks.")

    # Show sample chunk
    if split_doc_chunks:
        print("(*) Sample chunk:")
        print(f"\tContent: {split_doc_chunks[0].page_content[:200]}...")
        print(f"\tMetadata: {split_doc_chunks[0].metadata}")
    return split_doc_chunks

chunks = split_documents(documents=all_pdf_documents)

(*) Split 6 into 18 chunks.
(*) Sample chunk:
	Content: Amazon Kinesis Interview Questions and Answers
Interview-style questions covering Kinesis Data Streams, Kinesis Data Firehose, shards, partition keys,
consumer patterns, and how Kinesis compares to ot...
	Metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-08-27T11:17:36+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-08-27T11:17:36+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf_files\\amazon_kinesis_interview_qna.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'amazon_kinesis_interview_qna.pdf', 'file_type': 'pdf', 'processed_by': 'Shubhanshu'}


In [6]:
chunks[:5]

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-08-27T11:17:36+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-08-27T11:17:36+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf_files\\amazon_kinesis_interview_qna.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'amazon_kinesis_interview_qna.pdf', 'file_type': 'pdf', 'processed_by': 'Shubhanshu'}, page_content="Amazon Kinesis Interview Questions and Answers\nInterview-style questions covering Kinesis Data Streams, Kinesis Data Firehose, shards, partition keys,\nconsumer patterns, and how Kinesis compares to other streaming platforms.\nQ1. What is Amazon Kinesis?\nAmazon Kinesis is a family of managed services for collecting, processing, and analyzing real-time streaming\ndata at scale. It includes Kinesis Data Streams, Kinesis Data Firehose, and Kinesis Data Analytics, eac

### RAG PIPELINE - Embedding And Vector DB

In [7]:
import os
import numpy as np
from sentence_transformers import SentenceTransformer  # to use open-source Embedding model for sentences
import chromadb  # open-source Vector DB
from chromadb.config import Settings
import uuid  # to provide unique identifier to records
from typing import List, Dict, Any, Tuple, Optional
from sklearn.metrics.pairwise import cosine_similarity  # method to use for Retrieval
from langchain_core.documents.base import Document
import ollama

In [8]:
# %%bash
# pip install ollama

In [9]:
class EmbeddingManager:
    """
    Loads a text-embedding model and generates embeddings for input text.

    Two backends are supported:
        - SentenceTransformer: downloads and runs a HuggingFace
          sentence-embedding model locally (default).
        - Ollama: calls a locally running Ollama server to generate
          embeddings, avoiding any network call to HuggingFace.

    Attributes:
        model_name: Name or tag of the embedding model to load.
        model: Loaded SentenceTransformer instance when `ollama_enabled`
            is False; otherwise None (Ollama has no local model object).
        embedding_dim: Dimensionality of the embeddings produced by the
            loaded model. Set once the model has loaded successfully.
        ollama_enabled: Whether embeddings are generated via Ollama
            instead of SentenceTransformer.
    """

    def __init__(self, model_name: str = "all-MiniLM-L6-v2", use_ollama: bool = False) -> None:
        """
        Initialize the EmbeddingManager and load the underlying model.

        Args:
            model_name: Embedding model to load. When `use_ollama` is
                False, this should be a HuggingFace SentenceTransformer
                model name (e.g. "all-MiniLM-L6-v2"). When `use_ollama`
                is True, this should be an Ollama model tag that has
                already been pulled locally (e.g. "all-minilm").
            use_ollama: If True, generate embeddings via a local Ollama
                server instead of downloading a SentenceTransformer model.

        Raises:
            Exception: Re-raised if the underlying model fails to load.
                See `__load_model` for details.
        """
        self.model_name = model_name
        self.model: Optional[SentenceTransformer] = None
        self.embedding_dim: Optional[int] = None
        self.ollama_enabled = use_ollama
        self.__load_model()
        print(f"(*) EmbeddingManager initialized")

    def __load_model(self) -> None:
        """
        Load the embedding backend (Ollama or SentenceTransformer).

        For Ollama, sends a single test embedding request to confirm the
        model is reachable and to determine its embedding dimension. For
        SentenceTransformer, downloads (if needed) and loads the model
        into memory.

        Raises:
            Exception: Any exception raised while loading the model is
                printed and re-raised so the caller knows initialization
                failed rather than silently ending up with a half-built
                object.
        """
        try:
            print(f"(*) Loading embedding model: {self.model_name} model.")
            if self.ollama_enabled:
                test_response = ollama.embed(model=self.model_name, input="test")
                self.embedding_dim = len(test_response["embeddings"][0])
            else:
                self.model = SentenceTransformer(self.model_name)
                self.embedding_dim = self.model.get_embedding_dimension()
            print(f"(*) Model loaded successfully. Embedding dimension: {self.embedding_dim}")
        except Exception as ex:
            print(f"(*) Error loading {self.model_name} model -- {ex}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts.

        Args:
            texts: List of text strings to embed.

        Returns:
            Array of shape (len(texts), embedding_dim) containing one
            embedding vector per input text.

        Raises:
            ValueError: If the model has not been loaded successfully
                (i.e. `embedding_dim` is still None).
        """
        if self.embedding_dim is None:
            raise ValueError("Model not loaded")
        if texts and isinstance(texts[0], Document):
            texts = [t.page_content if isinstance(t, Document) else t for t in texts]
            print(f"(*) Found List[Document] as input so coverted to List[str] -- considering only the page_content.")
        print(f"(*) Generating embedding for {len(texts)} texts.")
        if self.ollama_enabled:
            response = ollama.embed(model=self.model_name, input=texts)
            embeddings = np.array(response["embeddings"])
        else:
            embeddings = self.model.encode(texts, show_progress_bar=True)

        print(f"(*) Generated embeddings with shape: {embeddings.shape}")
        return embeddings

    def __repr__(self) -> str:
        """
        Return a concise, informative representation of this instance.

        Returns:
            A string showing the backend in use, model name, and
            embedding dimension, useful for quick inspection in a
            notebook cell (e.g. just typing `embedding_manager`).
        """
        backend = "Ollama" if self.ollama_enabled else "SentenceTransformer"
        return (
            f"EmbeddingManager(backend={backend}, model_name='{self.model_name}', "
            f"embedding_dim={self.embedding_dim})"
        )

> **Setup:** Before running the cell below, open a terminal and run:
> ```
> ollama pull all-minilm
> ```
> This only needs to be done once — it downloads the model locally so Ollama can serve embeddings without any HuggingFace network call.

In [10]:
# Initialize EmbeddingManager
embedding_manager = EmbeddingManager(model_name="all-minilm", use_ollama=True)
embedding_manager

(*) Loading embedding model: all-minilm model.
(*) Model loaded successfully. Embedding dimension: 384
(*) EmbeddingManager initialized


EmbeddingManager(backend=Ollama, model_name='all-minilm', embedding_dim=384)

In [11]:
type(chunks[0])

langchain_core.documents.base.Document

In [12]:
embedding_manager.generate_embeddings(texts=chunks)

(*) Found List[Document] as input so coverted to List[str] -- considering only the page_content.
(*) Generating embedding for 18 texts.
(*) Generated embeddings with shape: (18, 384)


array([[ 0.03721707, -0.04086503, -0.02754767, ...,  0.05597661,
        -0.02863583, -0.03132946],
       [ 0.02891317, -0.00748097, -0.01824933, ...,  0.03178722,
        -0.05728503, -0.02151792],
       [-0.05313619, -0.03713154, -0.00189085, ...,  0.02896781,
        -0.05339227,  0.00086601],
       ...,
       [-0.03222534, -0.05505302, -0.01302019, ..., -0.04394727,
        -0.03308165,  0.01561209],
       [-0.08385927,  0.00539647,  0.02707566, ..., -0.04393351,
         0.06572454,  0.01457635],
       [-0.0059295 ,  0.0234467 , -0.03517581, ..., -0.02224248,
         0.08390861,  0.02948227]], shape=(18, 384))

#### VectorStore

In [13]:
class VectorStore:
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self.__initialize_store()
        print(f"(*) VectorStore initialized.")

    def __initialize_store(self):
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embedding for RAG."}
            )
            print(f"(*) VectorDB connected. Collection: {self.collection_name}")
            print(f"(*) Existing documents in collection: {self.collection.count()}")
        except Exception as ex:
            print(f"(*) Error while initializing Vector Store -- {ex}")

    def ingest_documents(self, documents: List[Any], embeddings: np.ndarray):
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match the number of embeddings")
        print(f"(*) Ingestion {len(documents)} documents to the Vector Store.")

        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for idx, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generating unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:10]}_{idx}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = idx
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())

        # Ingest to Collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"(*) Successfully ingested {len(documents)} to the Vector Store.")
            print(f"(*) Total documents in collection: {self.collection.count()}")
        except Exception as ex:
            print(f"(*) Error while ingesting documents to the Vector Store -- {ex}")
            raise

In [14]:
vector_store = VectorStore()
vector_store

(*) VectorDB connected. Collection: pdf_documents
(*) Existing documents in collection: 36
(*) VectorStore initialized.


In [15]:
# Converting the List[Document] to List[str]
texts = list(map(lambda x: x.page_content, chunks))

# Generating embeddings from the chunk
embeddings = embedding_manager.generate_embeddings(texts=texts)

(*) Generating embedding for 18 texts.


(*) Generated embeddings with shape: (18, 384)


In [16]:
# Adding Chunks, Embeddings into VectorStore
vector_store.ingest_documents(documents=chunks, embeddings=embeddings)

(*) Ingestion 18 documents to the Vector Store.
(*) Successfully ingested 18 to the Vector Store.
(*) Total documents in collection: 54


### RAG PIPELINE - Retriever from Vector DB Pipeline

In [17]:
class RAGRetriever:
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
        print(f"(*) RAGRetriever initialized.")
    
    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0):
        print(f"(*) Retrieving documents for query: '{query}'")
        print(f"(*) Top K: {top_k},  Score Threshold: {score_threshold}")

        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        try:
            result = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            # Process results
            retrieved_docs = []

            if result['documents'] and result['documents'][0]:
                documents = result['documents'][0]
                metadatas = result['metadatas'][0]
                distances = result['distances'][0]
                ids = result['ids'][0]

                for idx, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': idx + 1
                        })
                
                print(f"(*) Retrieved {len(retrieved_docs)} documents (after filtering).")
            else:
                print("(*) No document found.")
            
            return retrieved_docs
        except Exception as ex:
            print(f"(*) Error while retrieving data for query '{query}' -- {ex}")
            return []

In [18]:
rag_retriever = RAGRetriever(vector_store=vector_store, embedding_manager=embedding_manager)
rag_retriever

(*) RAGRetriever initialized.


In [19]:
query = "How does Kinesis compare to Apache Kafka?"
result = rag_retriever.retrieve(query=query)
result

(*) Retrieving documents for query: 'How does Kinesis compare to Apache Kafka?'
(*) Top K: 5,  Score Threshold: 0.0
(*) Generating embedding for 1 texts.
(*) Generated embeddings with shape: (1, 384)
(*) Retrieved 5 documents (after filtering).


[{'id': 'doc_878ad1c65b_3',
  'content': 'balancing across shards, checkpointing progress, and responding to shard splits or merges, so the developer\ncan focus on record-processing logic.\nQ8. How does Kinesis compare to Apache Kafka?\nBoth are used for real-time event streaming, but Kinesis is a fully managed AWS service with pricing based on\nshards and requests, while Kafka is typically self-managed or run via a managed offering like MSK, giving\nmore control over configuration, retention, and partitioning at the cost of more operational responsibility.',
  'metadata': {'producer': 'ReportLab PDF Library - (opensource)',
   'source': '..\\data\\pdf_files\\amazon_kinesis_interview_qna.pdf',
   'author': '(anonymous)',
   'creationdate': '2026-08-27T11:17:36+00:00',
   'subject': '(unspecified)',
   'page': 0,
   'trapped': '/False',
   'title': '(anonymous)',
   'creator': '(unspecified)',
   'moddate': '2026-08-27T11:17:36+00:00',
   'total_pages': 1,
   'content_length': 516,
   '

___
___

### RAG PIPELINE - Integration of Vector DB Context Pipeline with LLM Output

In [20]:
### Simple RAG pipeline with Groq LLM
import os
from langchain_groq import ChatGroq
from langchain_ollama import ChatOllama  # Alternate to ChatGroq
from dotenv import load_dotenv

load_dotenv()

### Initialize the Groq LLM (set GROQ_API_KEY in environment)
GROQ_API_KEY = os.getenv('GROQ_API_KEY')

In [21]:
%pip install -U langchain-ollama

Note: you may need to restart the kernel to use updated packages.


##### NOTE: Run this once in a terminal before using the LLM, so the model is available locally: `ollama pull gemma2:9b`

In [22]:
use_ollama_llm = True
max_token = 1024
temperature = 0.1

if use_ollama_llm:
    model_name = "gemma2:9b"
    llm_model = ChatOllama(
        model=model_name,
        temperature=temperature,
        num_predict=max_token,  # Ollama's equivalent of Groq's max_tokens
    )
else:
    model_name = "gemma2-9b-it"
    llm_model = ChatGroq(
        groq_api_key=GROQ_API_KEY,
        model_name=model_name,
        temperature=temperature,
        max_tokens=max_token
    )
llm_model

ChatOllama(metadata={'lc_versions': {'langchain-core': '1.6.0', 'langchain': '1.3.17'}}, model='gemma2:9b', num_predict=1024, temperature=0.1)

In [23]:
SYSTEM_PROMPT = """Use the following context to answer the question concisely.
    CONTEXT:
    {}

    QUERY:
    {}

    ANSWER:
    """

In [24]:
def rag(query: str, retriever: RAGRetriever, llm: Any, top_k: int=3):
    results = retriever.retrieve(query=query, top_k=top_k)
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."

    response = llm.invoke([SYSTEM_PROMPT.format(context, query)])
    return response.content

In [26]:
question = "How does Kinesis compare to Apache Kafka?"
answer = rag(query=question, retriever=rag_retriever, llm=llm_model)
print(answer)

(*) Retrieving documents for query: 'How does Kinesis compare to Apache Kafka?'
(*) Top K: 3,  Score Threshold: 0.0
(*) Generating embedding for 1 texts.
(*) Generated embeddings with shape: (1, 384)
(*) Retrieved 3 documents (after filtering).
Both Kinesis and Apache Kafka are used for real-time event streaming. Kinesis is a fully managed AWS service, simpler to use but with less control over configuration. Kafka is typically self-managed or run via managed offerings like MSK, offering more control but requiring more operational responsibility.  



In [27]:
question = "What's my name?"
answer = rag(query=question, retriever=rag_retriever, llm=llm_model)
print(answer)

(*) Retrieving documents for query: 'What's my name?'
(*) Top K: 3,  Score Threshold: 0.0
(*) Generating embedding for 1 texts.
(*) Generated embeddings with shape: (1, 384)
(*) Retrieved 0 documents (after filtering).
No relevant context found to answer the question.


In [ ]:
def rag_advanced(query: str, retriever: RAGRetriever, llm: Any, top_k: int=3, min_score: float=0.2, return_context: bool=False):
    results = retriever.retrieve(query=query, top_k=top_k)
    if not results:
        return {
            "answer": "No relevant context found to answer the question.",
            "sources": [],
            "confidence": 0.0,
            "context": ""
        }
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc["similarity_score"] for doc in results])

    response = llm.invoke([SYSTEM_PROMPT.format(context, query)])
    output = {
        "answer": response.content,
        "sources": sources,
        "confidence": confidence
    }
    if return_context:
        output["context"] = context
    return output

In [29]:
question = "How does Kinesis perform as compared to Apache Kafka?"
result = rag_advanced(query=question, retriever=rag_retriever, llm=llm_model, top_k=5, min_score=0.1, return_context=True)
print(f"Answer: {result['answer']}")
print(f"Sources: {result['sources']}")
print(f"Confidence: {result['confidence']}")
print(f"Context Preview: {result['context'][:300]}")

(*) Retrieving documents for query: 'How does Kinesis perform as compared to Apache Kafka?'
(*) Top K: 5,  Score Threshold: 0.0
(*) Generating embedding for 1 texts.
(*) Generated embeddings with shape: (1, 384)
(*) Retrieved 5 documents (after filtering).
Answer: Kinesis is a fully managed service, simpler to set up and operate, but offers less control over configuration compared to Kafka. Kafka, often self-managed, provides more granular control but demands more operational overhead.  Both excel at real-time event streaming. 

Sources: [{'source': 'amazon_kinesis_interview_qna.pdf', 'page': 0, 'score': 0.6919056177139282, 'preview': 'balancing across shards, checkpointing progress, and responding to shard splits or merges, so the developer\ncan focus on record-processing logic.\nQ8. How does Kinesis compare to Apache Kafka?\nBoth are used for real-time event streaming, but Kinesis is a fully managed AWS service with pricing based o...'}, {'source': 'amazon_kinesis_interview_qna.pdf',

In [30]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

In [31]:
query = "Features of Kafka"
adv_rag = AdvancedRAGPipeline(rag_retriever, llm_model)
result = adv_rag.query(question=query, top_k=3, min_score=0.1, stream=True, summarize=True)
print("\n(*) Final Answer:", result['answer'])
print("(*) Summary:", result['summary'])
print("(*) History:", result['history'][-1])

(*) Retrieving documents for query: 'Features of Kafka'
(*) Top K: 3,  Score Threshold: 0.1
(*) Generating embedding for 1 texts.
(*) Generated embeddings with shape: (1, 384)
(*) Retrieved 3 documents (after filtering).
Streaming answer:
Use the following context to answer the question concisely.
Context:
Q8. What is Kafka Connect?
Kafka Connect is a framework for reliably streaming data between Kafka and external systems, such as
databases, file systems, or search indexes, using reusable source and sink connectors instead of writing
custom producer or consumer code.
Q9. What is Kafka Streams?
Kafka Streams is a client library for building stream-processing applications directly on top of Kafka. It
supports operations like filtering, aggregating, and joining streams, and treats state and processing as part of

Q8. What is Kafka Connect?
Kafka Connect is a framework for reliably streaming data between Kafka and external systems, such as
databases, file systems, or search indexes, using